In [ ]:
import sys
sys.path.insert(0, '../Results')
sys.path.insert(0, '../../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import json
import ast

from python_utils.visualization_utils import create_summary_table, plot_snr_performance

In [ ]:
if not os.path.exists("Figures"):
    os.mkdir("Figures")

In [ ]:
# ---------------------------------------------------------
# 0. Load results
# ---------------------------------------------------------
csv_path = "../Results/predictive_bss_antisparse_taylor_error_results_V3.csv"
RESULTS_DF = pd.read_csv(csv_path)

In [ ]:
# ---------------------------------------------------------
# 1. Robust parser for array-valued CSV fields
# ---------------------------------------------------------
def recover_array(x):
    if isinstance(x, np.ndarray):
        return x.astype(float)
    if isinstance(x, list):
        return np.asarray(x, dtype=float)
    if pd.isna(x):
        return np.array([], dtype=float)

    if isinstance(x, str):
        s = x.strip()
        try:
            return np.asarray(json.loads(s), dtype=float)
        except Exception:
            pass
        try:
            return np.asarray(ast.literal_eval(s), dtype=float)
        except Exception:
            pass
        raise ValueError(f"Could not parse array from string: {s[:120]}...")

    raise TypeError(f"Unsupported type for array recovery: {type(x)}")


# ---------------------------------------------------------
# 2. Recover only the needed array columns
# ---------------------------------------------------------
RESULTS_DF["actual_error_array"] = RESULTS_DF["actual_error_abs"].apply(recover_array)
RESULTS_DF["tight_bound_array"] = RESULTS_DF["tight_theoretical_bound"].apply(recover_array)

# ---------------------------------------------------------
# 3. Extract final converged values
# ---------------------------------------------------------
RESULTS_DF["final_actual_error"] = RESULTS_DF["actual_error_array"].apply(lambda x: x[-1])
RESULTS_DF["final_tight_bound"] = RESULTS_DF["tight_bound_array"].apply(lambda x: x[-1])

# ---------------------------------------------------------
# 4. Optional safety check: verify bound holds trajectory-wise
# ---------------------------------------------------------
tol = 1e-12
RESULTS_DF["tight_bound_valid"] = RESULTS_DF.apply(
    lambda row: np.all(row["actual_error_array"] <= row["tight_bound_array"] + tol),
    axis=1,
)
print("Tight bound valid for all trajectories?:", RESULTS_DF["tight_bound_valid"].all())

# ---------------------------------------------------------
# 5. Plot settings
# ---------------------------------------------------------
eps = 1e-16
record_interval = 1000  # adjust if needed

TITLE_FONTSIZE = 17
LABEL_FONTSIZE = 15
TICK_FONTSIZE = 13
LEGEND_FONTSIZE = 13
LINEWIDTH = 2.6
MARKERSIZE = 7


# ---------------------------------------------------------
# 6. Helper: time-evolution plot for a fixed rho
# ---------------------------------------------------------
def plot_mean_trajectory_for_rho(df, target_rho, save_path):
    df_rho = df[np.isclose(df["rho"].to_numpy(), target_rho)].copy()
    if df_rho.empty:
        raise ValueError(f"No rows found for rho = {target_rho}")

    mean_actual_error_traj = np.mean(np.vstack(df_rho["actual_error_array"].values), axis=0)
    mean_tight_bound_traj = np.mean(np.vstack(df_rho["tight_bound_array"].values), axis=0)

    iterations = np.arange(len(mean_actual_error_traj)) * record_interval

    plt.figure(figsize=(8, 5))
    plt.semilogy(
        iterations,
        mean_actual_error_traj + eps,
        label="Mean Actual Error",
        color="blue",
        lw=LINEWIDTH,
    )
    plt.semilogy(
        iterations,
        mean_tight_bound_traj + eps,
        label="Mean Theoretical Bound",
        color="red",
        linestyle="--",
        lw=LINEWIDTH,
    )

    plt.title(
        rf"Time-Evolution of Mean Surrogate Error ($\rho = {target_rho:.1f}$)",
        fontsize=TITLE_FONTSIZE,
        fontweight="bold",
    )
    plt.xlabel("Training Iterations", fontsize=LABEL_FONTSIZE)
    plt.ylabel("Error Magnitude (Log Scale)", fontsize=LABEL_FONTSIZE)
    plt.tick_params(axis="both", which="major", labelsize=TICK_FONTSIZE)
    plt.grid(True, which="both", ls="--", alpha=0.5)
    plt.legend(fontsize=LEGEND_FONTSIZE)
    plt.tight_layout()
    plt.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.show()


# ---------------------------------------------------------
# 7. Plot 1: Time-evolution for rho = 0
# ---------------------------------------------------------
plot_mean_trajectory_for_rho(
    RESULTS_DF,
    target_rho=0.0,
    save_path="Figures/Taylor_Analysis_uncorrelated.pdf",
)

# ---------------------------------------------------------
# 8. Plot 2: Time-evolution for rho = 0.4
# ---------------------------------------------------------
plot_mean_trajectory_for_rho(
    RESULTS_DF,
    target_rho=0.4,
    save_path="Figures/Taylor_Analysis_rho_0p4.pdf",
)

# ---------------------------------------------------------
# 9. Plot 3: Converged values vs rho
# ---------------------------------------------------------
grouped_results = (
    RESULTS_DF.groupby("rho")[["final_actual_error", "final_tight_bound"]]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.semilogy(
    grouped_results["rho"],
    grouped_results["final_actual_error"] + eps,
    marker="o",
    label="Mean Converged Actual Error",
    color="blue",
    lw=LINEWIDTH,
    markersize=MARKERSIZE,
)
plt.semilogy(
    grouped_results["rho"],
    grouped_results["final_tight_bound"] + eps,
    marker="s",
    label="Mean Converged Theoretical Bound",
    color="red",
    linestyle="--",
    lw=LINEWIDTH,
    markersize=MARKERSIZE,
)

plt.title(
    "Mean Surrogate Error at Convergence vs. Correlation",
    fontsize=TITLE_FONTSIZE,
    fontweight="bold",
)
plt.xlabel(r"Source Correlation ($\rho$)", fontsize=LABEL_FONTSIZE)
plt.ylabel("Error Magnitude (Log Scale)", fontsize=LABEL_FONTSIZE)
plt.tick_params(axis="both", which="major", labelsize=TICK_FONTSIZE)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=LEGEND_FONTSIZE)
plt.tight_layout()
plt.savefig("Figures/Taylor_Analysis_wrt_correlation.pdf", format="pdf", bbox_inches="tight")
plt.show()

# ---------------------------------------------------------
# 10. Plot 4: Scatter plot of actual error vs theoretical bound
#     Uses ALL recorded points from all trajectories
#     Since y-axis is the theoretical bound, valid points should lie on/above y = x
# ---------------------------------------------------------
all_actual = np.concatenate(RESULTS_DF["actual_error_array"].values) + eps
all_bound = np.concatenate(RESULTS_DF["tight_bound_array"].values) + eps
all_rho = np.concatenate([
    np.full(len(arr), rho, dtype=float)
    for arr, rho in zip(RESULTS_DF["actual_error_array"], RESULTS_DF["rho"])
])

line_min = min(all_actual.min(), all_bound.min())
line_max = max(all_actual.max(), all_bound.max())

plt.figure(figsize=(7.5, 6))
sc = plt.scatter(
    all_actual,
    all_bound,
    c=all_rho,
    cmap="viridis",
    s=18,
    alpha=0.45,
    edgecolors="none",
    rasterized=True,
)

plt.loglog(
    [line_min, line_max],
    [line_min, line_max],
    color="black",
    lw=2.0,
    label=r"$y=x$",
)

plt.xscale("log")
plt.yscale("log")
plt.title(
    "Actual Error vs. Theoretical Bound",
    fontsize=TITLE_FONTSIZE,
    fontweight="bold",
)
plt.xlabel("Actual Error", fontsize=LABEL_FONTSIZE)
plt.ylabel("Theoretical Bound", fontsize=LABEL_FONTSIZE)
plt.tick_params(axis="both", which="major", labelsize=TICK_FONTSIZE)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=LEGEND_FONTSIZE)

cbar = plt.colorbar(sc)
cbar.set_label(r"Source Correlation ($\rho$)", fontsize=LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=TICK_FONTSIZE)

plt.tight_layout()
plt.savefig("Figures/Taylor_Analysis_scatter_bound_validation.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------
# 1. Robust parser for array-valued CSV fields
# ---------------------------------------------------------
def recover_array(x):
    if isinstance(x, np.ndarray):
        return x.astype(float)
    if isinstance(x, list):
        return np.asarray(x, dtype=float)
    if pd.isna(x):
        return np.array([], dtype=float)

    if isinstance(x, str):
        s = x.strip()
        try:
            return np.asarray(json.loads(s), dtype=float)
        except Exception:
            pass
        try:
            return np.asarray(ast.literal_eval(s), dtype=float)
        except Exception:
            pass
        raise ValueError(f"Could not parse array from string: {s[:120]}...")

    raise TypeError(f"Unsupported type for array recovery: {type(x)}")

# ---------------------------------------------------------
# 2. Recover only the needed array columns
# ---------------------------------------------------------
RESULTS_DF["actual_error_array"] = RESULTS_DF["actual_error_abs"].apply(recover_array)
RESULTS_DF["tight_bound_array"] = RESULTS_DF["tight_theoretical_bound"].apply(recover_array)

# ---------------------------------------------------------
# 3. Extract final converged values
# ---------------------------------------------------------
RESULTS_DF["final_actual_error"] = RESULTS_DF["actual_error_array"].apply(lambda x: x[-1])
RESULTS_DF["final_tight_bound"] = RESULTS_DF["tight_bound_array"].apply(lambda x: x[-1])

# ---------------------------------------------------------
# 4. Optional safety check: verify bound holds trajectory-wise
# ---------------------------------------------------------
tol = 1e-12
RESULTS_DF["tight_bound_valid"] = RESULTS_DF.apply(
    lambda row: np.all(row["actual_error_array"] <= row["tight_bound_array"] + tol),
    axis=1,
)
print("Tight bound valid for all trajectories?:", RESULTS_DF["tight_bound_valid"].all())

# ---------------------------------------------------------
# 5. Plot 1: Time-evolution of the mean for rho = 0
# ---------------------------------------------------------
eps = 1e-16
record_interval = 1000  # adjust if needed

df_rho0 = RESULTS_DF[RESULTS_DF["rho"] == 0.0].copy()

mean_actual_error_traj = np.mean(np.vstack(df_rho0["actual_error_array"].values), axis=0)
mean_tight_bound_traj = np.mean(np.vstack(df_rho0["tight_bound_array"].values), axis=0)

iterations = np.arange(len(mean_actual_error_traj)) * record_interval

plt.figure(figsize=(8, 5))
plt.semilogy(
    iterations,
    mean_actual_error_traj + eps,
    label="Mean Actual Error",
    color="blue",
    lw=2.5,
)
plt.semilogy(
    iterations,
    mean_tight_bound_traj + eps,
    label="Mean Theoretical Bound",
    color="red",
    linestyle="--",
    lw=2.5,
)

plt.title(r"Time-Evolution of Mean Surrogate Error ($\rho = 0.0$)", fontsize=16, fontweight="bold")
plt.xlabel("Training Iterations", fontsize=14)
plt.ylabel("Magnitude (Log Scale)", fontsize=14)
plt.tick_params(axis="both", which="major", labelsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig("Figures/Taylor_Analysis_uncorrelated.pdf", format="pdf", bbox_inches="tight")
plt.show()

# ---------------------------------------------------------
# 6. Plot 2: Converged values vs rho
# ---------------------------------------------------------
grouped_results = RESULTS_DF.groupby("rho")[["final_actual_error", "final_tight_bound"]].mean().reset_index()

plt.figure(figsize=(8, 5))
plt.semilogy(
    grouped_results["rho"],
    grouped_results["final_actual_error"] + eps,
    marker="o",
    label="Mean Converged Actual Error",
    color="blue",
    lw=2.5,
    markersize=7,
)
plt.semilogy(
    grouped_results["rho"],
    grouped_results["final_tight_bound"] + eps,
    marker="s",
    label="Mean Converged Theoretical Bound",
    color="red",
    linestyle="--",
    lw=2.5,
    markersize=7,
)

plt.title("Mean Surrogate Error at Convergence vs. Correlation", fontsize=16, fontweight="bold")
plt.xlabel(r"Source Correlation ($\rho$)", fontsize=14)
plt.ylabel("Error Magnitude (Log Scale)", fontsize=14)
plt.tick_params(axis="both", which="major", labelsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig("Figures/Taylor_Analysis_wrt_correlation.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------
# 2. Recover all array columns
# ---------------------------------------------------------
RESULTS_DF["actual_error_array"] = RESULTS_DF["actual_error_abs"].apply(recover_array)
RESULTS_DF["signed_remainder_array"] = RESULTS_DF["actual_remainder_signed"].apply(recover_array)
RESULTS_DF["tight_bound_array"] = RESULTS_DF["tight_theoretical_bound"].apply(recover_array)
RESULTS_DF["loose_bound_array"] = RESULTS_DF["loose_theoretical_bound"].apply(recover_array)

RESULTS_DF["offdiag_frob_array"] = RESULTS_DF["off_diagonal_frob_norm"].apply(recover_array)
RESULTS_DF["B_frob_array"] = RESULTS_DF["B_frob_norm"].apply(recover_array)
RESULTS_DF["B_spec_array"] = RESULTS_DF["B_spectral_norm"].apply(recover_array)
RESULTS_DF["lambda_min_C_array"] = RESULTS_DF["lambda_min_C"].apply(recover_array)
RESULTS_DF["lambda_min_B_array"] = RESULTS_DF["lambda_min_B"].apply(recover_array)

# ---------------------------------------------------------
# 3. Extract final converged values
# ---------------------------------------------------------
RESULTS_DF["final_actual_error"] = RESULTS_DF["actual_error_array"].apply(lambda x: x[-1])
RESULTS_DF["final_signed_remainder"] = RESULTS_DF["signed_remainder_array"].apply(lambda x: x[-1])
RESULTS_DF["final_tight_bound"] = RESULTS_DF["tight_bound_array"].apply(lambda x: x[-1])
RESULTS_DF["final_loose_bound"] = RESULTS_DF["loose_bound_array"].apply(lambda x: x[-1])

RESULTS_DF["final_offdiag_frob"] = RESULTS_DF["offdiag_frob_array"].apply(lambda x: x[-1])
RESULTS_DF["final_B_frob"] = RESULTS_DF["B_frob_array"].apply(lambda x: x[-1])
RESULTS_DF["final_B_spec"] = RESULTS_DF["B_spec_array"].apply(lambda x: x[-1])
RESULTS_DF["final_lambda_min_C"] = RESULTS_DF["lambda_min_C_array"].apply(lambda x: x[-1])
RESULTS_DF["final_lambda_min_B"] = RESULTS_DF["lambda_min_B_array"].apply(lambda x: x[-1])

# Ratios to compare tightness of the bounds
eps = 1e-16
RESULTS_DF["final_tight_over_actual"] = (
    RESULTS_DF["final_tight_bound"] + eps
) / (RESULTS_DF["final_actual_error"] + eps)

RESULTS_DF["final_loose_over_actual"] = (
    RESULTS_DF["final_loose_bound"] + eps
) / (RESULTS_DF["final_actual_error"] + eps)

# ---------------------------------------------------------
# 4. Check bound validity trajectory-wise
# ---------------------------------------------------------
tol = 1e-12

RESULTS_DF["tight_bound_valid"] = RESULTS_DF.apply(
    lambda row: np.all(row["actual_error_array"] <= row["tight_bound_array"] + tol),
    axis=1,
)

RESULTS_DF["loose_bound_valid"] = RESULTS_DF.apply(
    lambda row: np.all(row["actual_error_array"] <= row["loose_bound_array"] + tol),
    axis=1,
)

print("Tight bound valid for all trajectories?:", RESULTS_DF["tight_bound_valid"].all())
print("Loose bound valid for all trajectories?:", RESULTS_DF["loose_bound_valid"].all())

# ---------------------------------------------------------
# 5. Plot 1: Time-evolution of the mean for rho = 0
# ---------------------------------------------------------
df_rho0 = RESULTS_DF[RESULTS_DF["rho"] == 0.0].copy()

mean_actual_error_traj = np.mean(np.vstack(df_rho0["actual_error_array"].values), axis=0)
mean_tight_bound_traj = np.mean(np.vstack(df_rho0["tight_bound_array"].values), axis=0)
mean_loose_bound_traj = np.mean(np.vstack(df_rho0["loose_bound_array"].values), axis=0)

# Optional: mean signed remainder, if you want to inspect cancellations
mean_signed_remainder_traj = np.mean(np.vstack(df_rho0["signed_remainder_array"].values), axis=0)

record_interval = 1000  # change if needed
iterations = np.arange(len(mean_actual_error_traj)) * record_interval

plt.figure(figsize=(8, 5))
plt.semilogy(
    iterations,
    mean_actual_error_traj + eps,
    label="Mean Actual Error",
    color="blue",
    lw=2.5,
)
plt.semilogy(
    iterations,
    mean_tight_bound_traj + eps,
    label="Mean Tight Theoretical Bound",
    color="red",
    linestyle="--",
    lw=2.5,
)
plt.semilogy(
    iterations,
    mean_loose_bound_traj + eps,
    label="Mean Loose Theoretical Bound",
    color="gray",
    linestyle=":",
    lw=2.5,
)

plt.title(r"Time-Evolution of Mean Surrogate Error ($\rho = 0.0$)", fontsize=16, fontweight="bold")
plt.xlabel("Training Iterations", fontsize=14)
plt.ylabel("Magnitude (Log Scale)", fontsize=14)
plt.tick_params(axis="both", which="major", labelsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()
# plt.savefig("Figures/Taylor_Analysis_uncorrelated_tight_vs_loose.pdf", format="pdf", bbox_inches="tight")
plt.show()


# ---------------------------------------------------------
# 6. Plot 2: Converged values vs rho
# ---------------------------------------------------------
grouped_results = RESULTS_DF.groupby("rho")[
    [
        "final_actual_error",
        "final_tight_bound",
        "final_loose_bound",
        "final_tight_over_actual",
        "final_loose_over_actual",
    ]
].mean().reset_index()

plt.figure(figsize=(8, 5))
plt.semilogy(
    grouped_results["rho"],
    grouped_results["final_actual_error"] + eps,
    marker="o",
    label="Mean Converged Actual Error",
    color="blue",
    lw=2.5,
    markersize=7,
)
plt.semilogy(
    grouped_results["rho"],
    grouped_results["final_tight_bound"] + eps,
    marker="s",
    label="Mean Converged Tight Bound",
    color="red",
    linestyle="--",
    lw=2.5,
    markersize=7,
)
plt.semilogy(
    grouped_results["rho"],
    grouped_results["final_loose_bound"] + eps,
    marker="^",
    label="Mean Converged Loose Bound",
    color="gray",
    linestyle=":",
    lw=2.5,
    markersize=7,
)

plt.title("Mean Surrogate Error at Convergence vs. Correlation", fontsize=16, fontweight="bold")
plt.xlabel(r"Source Correlation ($\rho$)", fontsize=14)
plt.ylabel("Error Magnitude (Log Scale)", fontsize=14)
plt.tick_params(axis="both", which="major", labelsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()
# plt.savefig("Figures/Taylor_Analysis_wrt_correlation_tight_vs_loose.pdf", format="pdf", bbox_inches="tight")
plt.show()


# ---------------------------------------------------------
# 7. Optional Plot 3: Tightness ratios vs rho
# ---------------------------------------------------------
plt.figure(figsize=(8, 5))
plt.semilogy(
    grouped_results["rho"],
    grouped_results["final_tight_over_actual"] + eps,
    marker="s",
    label="Tight / Actual",
    color="red",
    lw=2.5,
    markersize=7,
)
plt.semilogy(
    grouped_results["rho"],
    grouped_results["final_loose_over_actual"] + eps,
    marker="^",
    label="Loose / Actual",
    color="gray",
    linestyle="--",
    lw=2.5,
    markersize=7,
)

plt.title("Mean Bound-to-Error Ratio at Convergence", fontsize=16, fontweight="bold")
plt.xlabel(r"Source Correlation ($\rho$)", fontsize=14)
plt.ylabel("Bound / Actual Error (Log Scale)", fontsize=14)
plt.tick_params(axis="both", which="major", labelsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()
# plt.savefig("Figures/Taylor_Bound_Tightness_Ratio.pdf", format="pdf", bbox_inches="tight")
plt.show()


# ---------------------------------------------------------
# 8. Optional textual summary
# ---------------------------------------------------------
summary_table = RESULTS_DF.groupby("rho")[
    [
        "final_actual_error",
        "final_tight_bound",
        "final_loose_bound",
        "final_tight_over_actual",
        "final_loose_over_actual",
        "tight_bound_valid",
        "loose_bound_valid",
    ]
].agg(["mean", "std"])

print("\nSummary table:")
print(summary_table)

print("\nAverage final tight/actual ratio across all runs:",
      RESULTS_DF["final_tight_over_actual"].mean())
print("Average final loose/actual ratio across all runs:",
      RESULTS_DF["final_loose_over_actual"].mean())

In [ ]:
# import ast
# RESULTS_DF = df_predictiveBSS.copy()
# # ---------------------------------------------------------
# # 0. The Recovery Parser
# # ---------------------------------------------------------
# def recover_array(arr_str):
#     """Parses the string representation of a list-wrapped numpy array back into an actual array."""
#     if not isinstance(arr_str, str):
#         return arr_str # If it's already an array, just return it
    
#     # Strip the numpy-specific syntax to leave a standard Python list string
#     clean_str = arr_str.replace('[array([', '[').replace('])]', ']')
    
#     # ast.literal_eval safely parses the string (including the \n characters) into a list
#     parsed_list = ast.literal_eval(clean_str)
    
#     # Convert back to a numpy array
#     return np.array(parsed_list)

# # Recover the arrays!
# RESULTS_DF['actual_error_array'] = RESULTS_DF['actual_error'].apply(recover_array)
# RESULTS_DF['bound_array'] = RESULTS_DF['theoretical_bounds'].apply(recover_array)

# # Extract the final converged value (the last element of the array)
# RESULTS_DF['final_actual_error'] = RESULTS_DF['actual_error_array'].apply(lambda x: x[-1])
# RESULTS_DF['final_bound'] = RESULTS_DF['bound_array'].apply(lambda x: x[-1])


# # ---------------------------------------------------------
# # 1. Plot 1: Time-Evolution of the Mean (rho = 0.0)
# # ---------------------------------------------------------
# df_rho0 = RESULTS_DF[RESULTS_DF['rho'] == 0.0]

# # Stack the arrays vertically and calculate the mean across all seeds (axis=0)
# mean_actual_error_traj = np.mean(np.vstack(df_rho0['actual_error_array'].values), axis=0)
# mean_bound_traj = np.mean(np.vstack(df_rho0['bound_array'].values), axis=0)

# # Generate iterations array (Adjust the multiplier to match your recording interval!)
# record_interval = 1000 
# iterations = np.arange(len(mean_actual_error_traj)) * record_interval

# plt.figure(figsize=(8, 5))
# # Adding a tiny epsilon prevents log(0) warnings
# plt.semilogy(iterations, mean_actual_error_traj + 1e-16, 
#              label='Mean Actual Error', color='blue', lw=2.5)
# plt.semilogy(iterations, mean_bound_traj + 1e-16, 
#              label='Mean Theoretical Bound', color='red', linestyle='--', lw=2.5)

# # --- INCREASED FONT SIZES HERE ---
# plt.title(r'Time-Evolution of Mean Surrogate Error ($\rho = 0.0$)', fontsize=16, fontweight='bold')
# plt.xlabel('Training Iterations', fontsize=14)
# plt.ylabel('Magnitude (Log Scale)', fontsize=14)
# plt.tick_params(axis='both', which='major', labelsize=12) # Increases tick number sizes
# # ---------------------------------

# plt.grid(True, which="both", ls="--", alpha=0.5)
# plt.legend(fontsize=12)
# plt.tight_layout()
# plt.savefig('Figures/Taylor_Analysis_uncorrelated.pdf', format='pdf', bbox_inches = 'tight')
# plt.show()


# # ---------------------------------------------------------
# # 2. Plot 2: Convergence values vs. Rho
# # ---------------------------------------------------------
# # Group by rho and calculate the mean of the final values across all seeds
# grouped_results = RESULTS_DF.groupby('rho')[['final_actual_error', 'final_bound']].mean().reset_index()

# plt.figure(figsize=(8, 5))
# plt.semilogy(grouped_results['rho'], grouped_results['final_actual_error'] + 1e-16, 
#              marker='o', label='Mean Converged Actual Error', color='blue', lw=2.5, markersize=8)
# plt.semilogy(grouped_results['rho'], grouped_results['final_bound'] + 1e-16, 
#              marker='s', label='Mean Converged Theoretical Bound', color='red', linestyle='--', lw=2.5, markersize=8)

# # --- INCREASED FONT SIZES HERE ---
# plt.title('Mean Surrogate Error at Convergence vs. Correlation', fontsize=16, fontweight='bold')
# plt.xlabel(r'Source Correlation ($\rho$)', fontsize=14)
# plt.ylabel('Error Magnitude (Log Scale)', fontsize=14)
# plt.tick_params(axis='both', which='major', labelsize=12) # Increases tick number sizes
# # ---------------------------------

# plt.grid(True, which="both", ls="--", alpha=0.5)
# plt.legend(fontsize=12)
# plt.tight_layout()
# plt.savefig('Figures/Taylor_Analysis_wrt_correlation.pdf', format='pdf', bbox_inches = 'tight')
# plt.show()